In [1]:
from pathlib import Path
import numpy as np
import SimpleITK as sitk

from sam2.build_sam import build_sam2_video_predictor_npz
from App_utils.inference import run_medsam2_inference_from_arrays
from App_utils.prompt_utils import load_mask_like_reference
from App_utils.post_processing import (
    keep_largest_3d_connected_component,
    save_mask_like_reference,
)

from uncertainty_handling import determine_unc_thr, generate_and_save_ray_prompts
from eval import compute_metrics 

def self_prompting_pipeline(
    input_folder,
    volume_of_interest="CTVT",
    checkpoint="checkpoints/MedSAM2_latest.pt",
    cfg="configs/sam2.1_hiera_t512.yaml",
    verbose=False,
    eval = True,
    save_report = False,
):
    """Run MEDSAM2 for all subjects using uncertainty-generated point prompts + dense mask prompt."""

    root = Path(input_folder)
    if not root.is_dir():
        raise FileNotFoundError(f"Input folder does not exist: {root}")

    subjects = sorted([p.name for p in root.iterdir() if p.is_dir()])
    if len(subjects) == 0:
        raise ValueError(f"No subject folders found in: {root}")

    # Build predictor once (same strategy as Segmenter_App).
    if verbose:
        print(f"Building SAM predictor from checkpoint: {checkpoint}")

    predictor = build_sam2_video_predictor_npz(cfg, checkpoint)

    all_outputs = []
    all_performance_metrics = []

    report_path = root / "pipeline_report_nnUnet.txt"
    if save_report:
        with open(report_path, "w") as f:
            f.write("Self-prompting pipeline report\n")
            f.write(f"Input folder: {root}\n")
            f.write(f"Volume of Interest: {volume_of_interest}\n")
            f.write("=" * 60 + "\n")

    for num, subject in enumerate(subjects, start=1):

        subject_path = root / subject / "MR_StorT2"
        img_path = subject_path / "image.nii.gz"

        if not img_path.is_file():
            if verbose:
                print(f"[{num}/{len(subjects)}] Skipping {subject}: missing image")
            continue

        if volume_of_interest == "CTVT":
            dense_prompt_path = subject_path / "nnUNetOutput" / "mask_CTVT_427_nnUNet.nii.gz"
            unc_map_path      = subject_path / "nnUNetOutput" / "mask_CTVT_427_nnUNet_uncertaintyMap.nii.gz"
            gt_path           = subject_path / "mask_CTVT_427.nii.gz"
        elif volume_of_interest == "rectum":
            dense_prompt_path = subject_path / "nnUNetOutput" / "mask_Rectum_nnUNet.nii.gz"
            unc_map_path      = subject_path / "nnUNetOutput" / "mask_Rectum_nnUNet_uncertaintyMap.nii.gz"
            gt_path           = subject_path / "mask_Rectum.nii.gz"
        else:
            raise ValueError("volume_of_interest must be 'CTVT' or 'rectum'")

        if not dense_prompt_path.is_file():
            if verbose:
                print(f"[{num}/{len(subjects)}] Skipping {subject}: missing dense prompt {dense_prompt_path.name}")
            continue

        if verbose:
            print(f"[{num}/{len(subjects)}] Processing {subject}")

        img_itk = sitk.ReadImage(str(img_path))
        img = sitk.GetArrayFromImage(img_itk).astype(np.float32)

        spacing_sitk = img_itk.GetSpacing()  # (x, y, z)
        spacing = spacing_sitk[::-1]

        dense_mask_3d = load_mask_like_reference(str(dense_prompt_path), img.shape)

        gt_mask = load_mask_like_reference(str(gt_path), img.shape) if gt_path.is_file() else None

        unc_map_itk = sitk.ReadImage(str(unc_map_path)) if unc_map_path.is_file() else None
        unc_map = sitk.GetArrayFromImage(unc_map_itk).astype(np.float32) if unc_map_itk is not None else None


        if eval:
            performance_metrics = compute_metrics(dense_mask_3d, gt_mask, spacing=spacing, surface_dice_tol=2.0, apl_tol=0.0)
            all_performance_metrics.append(performance_metrics)

        if save_report and eval:
            with open(report_path, "a") as f:
                f.write("-" * 60 + "\n")
                f.write(f"Subject: {subject}\n")
                f.write(f"Volume of Interest: {volume_of_interest}\n")
                f.write("Performance Metrics:\n")
                for metric_name, metric_value in performance_metrics.items():
                    f.write(f"  {metric_name}: {metric_value:.4f}\n")
                f.write("\n")


    return all_performance_metrics


In [ ]:
all_performance_metrics = self_prompting_pipeline(
    input_folder=r"C:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\data\LUNDPROBE\ExtendedSamples\development",
    volume_of_interest="CTVT",
    checkpoint="checkpoints/MedSAM2_latest.pt",
    cfg="configs/sam2.1_hiera_t512.yaml",
    verbose=False,
    eval = True,
    save_report = True,
)

c:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\sam2\modeling\sam\transformer.py:23: UserWarning: Flash Attention is disabled as it requires a GPU with Ampere (8.0) CUDA capability.
  OLD_GPU, USE_FLASH_ATTN, MATH_KERNEL_ON = get_sdpa_settings()


: 

In [ ]:
from pathlib import Path
import numpy as np
import SimpleITK as sitk

from App_utils.prompt_utils import load_mask_like_reference
from eval import compute_metrics

def evaluate_existing_nnunet_segmentations(
    input_folder,
    volume_of_interest="CTVT",
    save_report=True,
    report_name="nnunet_eval_report.txt",
    verbose=True,
):
    """Evaluate existing nnUNet segmentations against GT masks for all subjects."""

    root = Path(input_folder)
    if not root.is_dir():
        raise FileNotFoundError(f"Input folder does not exist: {root}")

    subjects = sorted([p.name for p in root.iterdir() if p.is_dir()])
    if len(subjects) == 0:
        raise ValueError(f"No subject folders found in: {root}")

    all_metrics = []
    report_path = root / report_name

    if save_report:
        with open(report_path, "w") as f:
            f.write("nnUNet evaluation report\n")
            f.write(f"Input folder: {root}\n")
            f.write(f"Volume of Interest: {volume_of_interest}\n")
            f.write("=" * 60 + "\n")

    for num, subject in enumerate(subjects, start=1):
        subject_path = root / subject / "MR_StorT2"
        img_path = subject_path / "image.nii.gz"

        if volume_of_interest == "CTVT":
            pred_path = subject_path / "nnUNetOutput" / "mask_CTVT_427_nnUNet.nii.gz"
            gt_path = subject_path / "mask_CTVT_427.nii.gz"
        elif volume_of_interest == "rectum":
            pred_path = subject_path / "nnUNetOutput" / "mask_Rectum_nnUNet.nii.gz"
            gt_path = subject_path / "mask_Rectum.nii.gz"
        else:
            raise ValueError("volume_of_interest must be 'CTVT' or 'rectum'")

        if not img_path.is_file() or not pred_path.is_file() or not gt_path.is_file():
            if verbose:
                print(
                    f"[{num}/{len(subjects)}] Skipping {subject}: missing one of image/pred/gt"
                )
            continue

        img_itk = sitk.ReadImage(str(img_path))
        img = sitk.GetArrayFromImage(img_itk).astype(np.float32)
        spacing = img_itk.GetSpacing()[::-1]

        pred_mask = load_mask_like_reference(str(pred_path), img.shape)
        gt_mask = load_mask_like_reference(str(gt_path), img.shape)

        pred_bin = (pred_mask > 0).astype(np.uint8)
        gt_bin = (gt_mask > 0).astype(np.uint8)

        metrics = compute_metrics(
            pred_bin,
            gt_bin,
            spacing=spacing,
            surface_dice_tol=2.0,
            apl_tol=0.0,
        )
        metrics["Subject"] = subject
        all_metrics.append(metrics)

        if verbose:
            print(
                f"[{num}/{len(subjects)}] {subject} -> "
                f"HD95: {metrics['HD95_mm']:.3f}, "
                f"ASSD: {metrics['ASSD_mm']:.3f}, "
                f"SurfaceDice@2.0mm: {metrics['SurfaceDice@2.0mm']:.3f}"
            )

        if save_report:
            with open(report_path, "a") as f:
                f.write("-" * 60 + "\n")
                f.write(f"Subject: {subject}\n")
                for metric_name, metric_value in metrics.items():
                    if metric_name == "Subject":
                        continue
                    f.write(f"  {metric_name}: {metric_value:.4f}\n")
                f.write("\n")

    if len(all_metrics) == 0:
        raise ValueError("No subjects were evaluated. Check input paths and file names.")

    metric_keys = [k for k in all_metrics[0].keys() if k != "Subject"]
    summary = {}
    for k in metric_keys:
        vals = np.array([m[k] for m in all_metrics], dtype=float)
        summary[k] = {
            "mean": float(np.nanmean(vals)),
            "std": float(np.nanstd(vals)),
        }

    if verbose:
        print("\nAggregate metrics (mean +- std):")
        for k, v in summary.items():
            print(f"{k}: {v['mean']:.4f} +- {v['std']:.4f}")

    if save_report:
        with open(report_path, "a") as f:
            f.write("=" * 60 + "\n")
            f.write("Aggregate metrics (mean +- std)\n")
            for k, v in summary.items():
                f.write(f"{k}: {v['mean']:.4f} +- {v['std']:.4f}\n")

    return all_metrics, summary


nnunet_metrics, nnunet_summary = evaluate_existing_nnunet_segmentations(
    input_folder=r"C:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\data\LUNDPROBE\ExtendedSamples\development",
    volume_of_interest="CTVT",
    save_report=True,
    report_name="nnunet_eval_report.txt",
    verbose=True,
)

[1/25] newAcq_050f229dc2bdb64c -> HD95: 3.427, ASSD: 1.099, SurfaceDice@2.0mm: 0.750
[2/25] newAcq_0b4940fa31a1d650 -> HD95: 5.910, ASSD: 1.947, SurfaceDice@2.0mm: 0.587
[3/25] newAcq_0cc559a8bd82a14a -> HD95: 2.965, ASSD: 0.977, SurfaceDice@2.0mm: 0.826
[4/25] newAcq_1b911d6cb2348f30 -> HD95: 3.018, ASSD: 1.139, SurfaceDice@2.0mm: 0.720
[5/25] newAcq_1e0f8b9b01ce5f0b -> HD95: 3.570, ASSD: 1.177, SurfaceDice@2.0mm: 0.742
[6/25] newAcq_250d6075dd465a1a -> HD95: 4.604, ASSD: 1.248, SurfaceDice@2.0mm: 0.783
[7/25] newAcq_433a8d44fddd5b7f -> HD95: 2.500, ASSD: 0.760, SurfaceDice@2.0mm: 0.869
[8/25] newAcq_47ceabdbca398517 -> HD95: 2.500, ASSD: 0.645, SurfaceDice@2.0mm: 0.892
[9/25] newAcq_486b7494ee9d71e7 -> HD95: 1.796, ASSD: 0.413, SurfaceDice@2.0mm: 0.969
[10/25] newAcq_4a136e8fe320bd13 -> HD95: 5.000, ASSD: 1.628, SurfaceDice@2.0mm: 0.620
[11/25] newAcq_4cf11c1b68868daf -> HD95: 3.145, ASSD: 1.016, SurfaceDice@2.0mm: 0.808
[12/25] newAcq_4e322991cf1792d0 -> HD95: 2.525, ASSD: 0.750, Su